In [20]:
TEXT_FOLDER = "core_clinical_short/"
TENSOR_FOLDER = "core_clinical_short_gemma_27b/"
DIST_CSV = '74.short_dist.csv'

In [21]:
import os
from itertools import combinations

from pted import pted
from scipy.stats import wasserstein_distance_nd
from torch import load
from tqdm.autonotebook import tqdm

from utils import *

In [22]:
text_files = os.listdir(TEXT_FOLDER)
NEW = False

In [23]:
layers = range(63)

In [24]:
if NEW:
    dist = DataFrame(columns=['dist_type', 'layer', 'text1', 'text2', 'symptom', 'distance'])

    tokens = {}  # {text: [tokens]}
    residual = {}  # {text: [layer: <tensor>]}

    for t in text_files:
        # remove all <bos>
        residual[t] = [te[1:] for te in list(load(f'{TENSOR_FOLDER}{t}.pt'))]
    # Calculate Unit Vector of all residual streams for every layer.
    unit = {}  # {text: {layer: <tensor>}}

    for text, tensors in residual.items():
        unit[text] = [normalize(t) for t in tensors]

    # Calculate Centroid of all residual streams for every layer.
    cent = {}  # {text: {layer: <tensor>}}
    cent_unit = {}  # {text: {layer: <tensor>}}

    for text, tensors in residual.items():
        cent[text] = [centroid(t) for t in tensors]

    for text, tensors in unit.items():
        cent_unit[text] = [centroid(t) for t in tensors]

In [25]:


if NEW:
    # Generate matrix for 8 centroid distances
    dist = DataFrame(columns=['dist_type', 'layer', 'text1', 'text2', 'symptom', 'distance'])
    for f1, f2 in tqdm(combinations(text_files, r=2)):
        symptom = f1.split(',')[1] if f1.split(',')[1] == f2.split(',')[1] else None
        for l in layers:
            # Centroid Cosine
            dist.loc[len(dist)] = ['centroid_cosine', l, f1, f2, symptom, cos_dist(cent[f1][l], cent[f2][l])]
    
            # Centroid Cosine (Unit)
            dist.loc[len(dist)] = ['centroid_cosine_unit', l, f1, f2, symptom, cos_dist(cent_unit[f1][l], cent_unit[f2][l])]
    
            # Centroid Euclidean
            dist.loc[len(dist)] = ['centroid_euclidean', l, f1, f2, symptom, euc(cent[f1][l], cent[f2][l])]
    
            # Centroid Euclidean (Unit)
            dist.loc[len(dist)] = ['centroid_euclidean_unit', l, f1, f2, symptom, euc(cent_unit[f1][l], cent_unit[f2][l])]
    
            # Earth mover
            dist.loc[len(dist)] = ['earth_mover', l, f1, f2, symptom,
                                   wasserstein_distance_nd(residual[f1][l].detach().cpu().numpy(),
                                                           residual[f2][l].detach().cpu().numpy())]
    
            # Earth mover (Unit)
            dist.loc[len(dist)] = ['earth_mover_unit', l, f1, f2, symptom,
                                   wasserstein_distance_nd(unit[f1][l].detach().cpu().numpy(),
                                                           unit[f2][l].detach().cpu().numpy())]
    
            # Energy
            energy, _, _ = pted(residual[f1][l], residual[f2][l], return_all=True, two_tailed=False, permutations=1)
            dist.loc[len(dist)] = ['energy', l, f1, f2, symptom, energy]
    
            # Energy Unit
            energy_unit, _, _ = pted(unit[f1][l], unit[f2][l], return_all=True, two_tailed=False, permutations=1)
            dist.loc[len(dist)] = ['energy_unit', l, f1, f2, symptom, energy_unit]

    dist.to_csv(DIST_CSV, index=True)

## Part 2 — Separation experiment

In [26]:
import plotly.graph_objects as go
from pandas import DataFrame, read_csv
from plotly.subplots import make_subplots
from skbio.stats.distance import DistanceMatrix, anosim, permanova, permdisp
from tqdm.auto import tqdm
import warnings

In [27]:
# Constants
dist = read_csv(DIST_CSV, index_col=0)
LAYERS = sorted(dist['layer'].unique())
PERMU = 9999
dist_types = dist['dist_type'].unique()
NEW = False  # Calcuate everything the first time, otherwise load from previous results

In [28]:
labels = os.listdir(TEXT_FOLDER)
labels.sort()
grouping = [f.split(',')[1] for f in labels]

In [29]:
if NEW:
    anosim_rows = []
    permanova_rows = []
    permdisp_rows = []

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')

        for dt in tqdm(dist_types):
            for l in LAYERS:
                # Reconstruct the original distance matrix
                original = dist[(dist['layer'] == l) & (dist['dist_type'] == dt)]
                square = (original
                          .pivot(index='text1', columns='text2', values='distance')
                          .reindex(index=labels, columns=labels))
                square = square.combine_first(square.T).fillna(0.0)

                raw_mat = square.values.copy()
                dm = DistanceMatrix(raw_mat, ids=labels)

                anosim_result = anosim(dm, grouping, permutations=PERMU)
                R = anosim_result['test statistic']
                anosim_p = anosim_result['p-value']

                permanova_result = permanova(dm, grouping, permutations=PERMU)
                F = permanova_result['test statistic']
                permanova_p = permanova_result['p-value']

                permdisp_result = permdisp(dm, grouping, test="centroid", permutations=PERMU)
                F2 = permdisp_result['test statistic']
                permdisp_p = permdisp_result['p-value']

                anosim_rows.append([dt, l, R, anosim_p])
                permanova_rows.append([dt, l, F, permanova_p])
                permdisp_rows.append([dt, l, F2, permdisp_p])

        anosim_pd = DataFrame(anosim_rows, columns=['dist_type', 'layer', 'R', 'p-value'])
        permanova_pd = DataFrame(permanova_rows, columns=['dist_type', 'layer', 'pseudo-F', 'p-value'])
        permdisp_pd = DataFrame(permdisp_rows, columns=['dist_type', 'layer', 'F-value', 'p-value'])

In [30]:
# Save a separate file for ablation test
if NEW:
    permanova_pd.to_csv('74.permanova.csv', index=True)
    anosim_pd.to_csv('74.anosim_pd.csv', index=True)
    permdisp_pd.to_csv('74.permdisp.csv', index=True)
else:
    permanova_pd = read_csv('74.permanova.csv')
    anosim_pd = read_csv('74.anosim_pd.csv')
    permdisp_pd = read_csv('74.permdisp.csv')

In [31]:
FORMAL_TITLES = {
    'centroid_cosine': "Centroid Cosine<br>(raw)",
    'centroid_cosine_unit': "Centroid Cosine<br>(unit-normalized)",
    'centroid_euclidean': "Centroid Euclidean<br>(raw)",
    'centroid_euclidean_unit': "Centroid Euclidean<br>(unit-normalized)",
    'earth_mover': "Earth Mover's<br>(raw)",
    'earth_mover_unit': "Earth Mover's<br>(unit-normalized)",
    'energy': "Energy<br>(raw)",
    'energy_unit': "Energy<br>(unit-normalized)"
}

In [39]:
# ── Gate-first per-layer figure ────────────────────────────────────────────
# PERMDISP defines the eligible region at each layer (non-significant within-
# group dispersion). The operating peak is the maximum PERMANOVA pseudo-F WITHIN
# that region; a metric's global peak that lands on an ineligible (PERMDISP-
# significant) layer is shown but excluded. Colours match Table 1.
COLOR_PRIMARY   = "#000000"                 # black — pseudo-F trajectory
COLOR_REF       = "rgba(150,150,150,0.4)"   # reference line at y=0
COLOR_HIGHLIGHT = "#009E73"                 # green — peak markers (shared with ANOSIM supplement)
GATE_PASS  = "#009E73"                      # green — operating (eligible) peak
GATE_FAIL  = "#9a9a9a"                      # gray  — excluded (ineligible) peak
INADM_FILL = "rgba(213,94,0,0.25)"         # pale red — ineligible layer band
ALPHA = 0.05

metrics = [dt for dt in FORMAL_TITLES if dt in set(permanova_pd['dist_type'])]


def _sig_runs(layers_sorted):
    """Contiguous runs of PERMDISP-significant layers -> [[start, end], ...]."""
    runs = []
    for L in layers_sorted:
        if runs and L == runs[-1][1] + 1:
            runs[-1][1] = L
        else:
            runs.append([L, L])
    return runs


def _annotation_offset(m, Lo, Fo, Y_MAX, window=8):
    """Return (ax, ay) pixel offsets that avoid overlapping the line.

    Horizontal: offset toward whichever side of the peak has the larger
    average F drop (more visual clearance there).
    Vertical: go above unless the peak is within 1 F-unit of Y_MAX.
    """
    left_f  = m.loc[(m['layer'] >= Lo - window) & (m['layer'] < Lo),  'F']
    right_f = m.loc[(m['layer'] >  Lo)          & (m['layer'] <= Lo + window), 'F']
    left_drop  = Fo - (left_f.mean()  if len(left_f)  else Fo)
    right_drop = Fo - (right_f.mean() if len(right_f) else Fo)
    ax = -52 if left_drop > right_drop else 52   # toward the steeper drop
    ay = 53 if Fo > Y_MAX - 1.0 else -10         # below if near top, else above
    return ax, ay


# Per-metric gate computation (uses the full per-layer PERMDISP surface)
info = {}
for dt in metrics:
    pm = (permanova_pd[permanova_pd['dist_type'] == dt][['layer', 'pseudo-F']]
          .rename(columns={'pseudo-F': 'F'}))
    pp = (permdisp_pd[permdisp_pd['dist_type'] == dt][['layer', 'p-value']]
          .rename(columns={'p-value': 'permdisp_p'}))
    m = pm.merge(pp, on='layer').sort_values('layer').reset_index(drop=True)

    g = m.loc[m['F'].idxmax()]                              # global pseudo-F peak
    adm = m[m['permdisp_p'] > ALPHA]                        # eligible layers
    op = adm.loc[adm['F'].idxmax()] if len(adm) else None   # operating peak
    global_adm = bool(g['permdisp_p'] > ALPHA)
    runs = _sig_runs(sorted(m.loc[m['permdisp_p'] <= ALPHA, 'layer'].tolist()))

    info[dt] = dict(m=m, g=g, op=op, global_adm=global_adm, runs=runs,
                    op_F=(float(op['F']) if op is not None else -1.0))

# Order panels by eligible peak pseudo-F descending
ordered = sorted(metrics, key=lambda d: -info[d]['op_F'])

rows, cols = 3, 3
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[FORMAL_TITLES.get(dt, dt) for dt in ordered],
    shared_xaxes=False, shared_yaxes=True,
    horizontal_spacing=0.04, vertical_spacing=0.16,
)

fig.update_layout(
    template='plotly_white',
    font=dict(family="Helvetica, Arial, sans-serif", size=10, color="black"),
    # No in-figure title: Nature Portfolio wants it in the caption, and at
    # size 12.5 it rendered ~8.9 pt at 180 mm, over the 7 pt body-text cap.
    # Trimmed to leave ~10 px of dead canvas on each edge (measured off the
    # exported PDF): the old l=55/r=20/t=28/b=94 left 26/29/12/26 px of blank
    # border, and the figure is placed at width=	extwidth, so that border was
    # just shrinking the panels. Panels are ~5% wider and ~6% taller now.
    # t stays generous: the two-line subplot titles sit right under the canvas
    # edge, and at t=24 they cleared it by only 0.09 in and read as cut off.
    # t=32 gives 0.17 in, vs 0.13 in before any of this.
    margin=dict(l=38, r=4, t=32, b=66),
    width=720, height=580,
    showlegend=True,
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.14, yanchor='top',
                font=dict(size=10), bgcolor='rgba(0,0,0,0)'),
)

fig.update_annotations(font=dict(size=10, color="black", family="Helvetica, Arial, sans-serif"))

# Legend proxies
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(symbol='square', color='rgba(213,94,0,0.20)', size=11),
    name='ineligible layer (PERMDISP p ≤ 0.05)'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(color=GATE_PASS, size=9),
    name='operating peak'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(symbol='circle-open', color=GATE_FAIL, size=9, line=dict(width=1.6)),
    name='excluded global peak'))

for i, dt in enumerate(ordered):
    row, col = i // cols + 1, i % cols + 1
    d = info[dt]
    m = d['m']

    # Ineligible bands drawn as filled traces BEFORE the line trace so they
    # sit behind it. Using go.Scatter instead of add_vrect avoids a CEF
    # compositing bug in PyCharm where layer='below' shapes miss the initial
    # paint and only appear after a forced repaint (e.g. hovering another window).
    Y_MAX = 8
    for (a, b) in d['runs']:
        fig.add_trace(go.Scatter(
            x=[a - 0.5, a - 0.5, b + 0.5, b + 0.5, a - 0.5],
            y=[0, Y_MAX, Y_MAX, 0, 0],
            fill='toself', fillcolor=INADM_FILL,
            line=dict(width=0), mode='lines',
            showlegend=False, hoverinfo='skip',
        ), row=row, col=col)

    # Pseudo-F trajectory
    fig.add_trace(go.Scatter(
        x=m['layer'], y=m['F'], mode='lines',
        line=dict(color=COLOR_PRIMARY, width=1.4),
        customdata=m[['permdisp_p']],
        hovertemplate='layer=%{x}<br>pseudo-F=%{y:.2f}<br>PERMDISP p=%{customdata[0]:.4f}<extra></extra>',
        showlegend=False), row=row, col=col)

    # Reference line at y=0
    fig.add_hline(y=0, line_dash="dash", line_color=COLOR_REF, line_width=0.5,
                  row=row, col=col)

    # Excluded global peak — only when it falls on an ineligible layer
    if not d['global_adm']:
        fig.add_trace(go.Scatter(
            x=[d['g']['layer']], y=[d['g']['F']], mode='markers',
            marker=dict(symbol='circle-open', color=GATE_FAIL, size=8,
                        line=dict(width=1.6)),
            showlegend=False, hoverinfo='skip'), row=row, col=col)

    # Operating peak (green, no edge) + annotation
    if d['op'] is not None:
        Lo, Fo = int(d['op']['layer']), float(d['op']['F'])
        fig.add_trace(go.Scatter(
            x=[Lo], y=[Fo], mode='markers',
            marker=dict(color=GATE_PASS, size=7),
            showlegend=False, hoverinfo='skip'), row=row, col=col)
        ax_val, ay_val = _annotation_offset(m, Lo, Fo, Y_MAX)
        fig.add_annotation(
            x=Lo, y=Fo, text=f"F = {Fo:.2f}<br>L{Lo}",
            showarrow=True, arrowhead=2, arrowwidth=0.6, arrowsize=1,
            arrowcolor=GATE_PASS,
            font=dict(size=8, color=GATE_PASS, family="Helvetica, Arial, sans-serif"),
            bgcolor='rgba(255,255,255,0.85)', bordercolor='rgba(0,0,0,0.12)',
            borderwidth=0.5, align='center', ax=ax_val, ay=ay_val,
            row=row, col=col)

# Axis titles. Label the bottom-most OCCUPIED panel in each column: with 8
# panels in a 3x3 grid the row-3/col-3 cell is empty, so targeting `rows`
# blindly left the third column's axis unlabelled.
bottom_row_of_col = {}
for i in range(len(ordered)):
    bottom_row_of_col[i % cols + 1] = i // cols + 1
for c, r in sorted(bottom_row_of_col.items()):
    fig.update_xaxes(title_text="Transformer Layer", title_font=dict(size=10),
                     row=r, col=c)
fig.update_yaxes(title_text="PERMANOVA pseudo-F", title_font=dict(size=10),
                 row=2, col=1)

# Axis styling
fig.update_xaxes(tickfont=dict(size=9), showgrid=False, linecolor='black',
                 linewidth=0.6, ticks='outside', ticklen=3, tickwidth=0.6)
fig.update_yaxes(range=[0, Y_MAX], tickfont=dict(size=9), showgrid=True,
                 gridcolor='rgba(200,200,200,0.3)', gridwidth=0.5,
                 linecolor='black', linewidth=0.6, ticks='outside',
                 ticklen=3, tickwidth=0.6)

fig.show()


In [ ]:
# ── Export Fig. 2 for the manuscript ─────────────────────────────────
# npj Digital Medicine (Nature Portfolio) wants .pdf/.eps with non-outlined
# editable text and embedded fonts, which is what Kaleido writes. PDF over EPS
# because EPS cannot carry the alpha in the shaded bands and annotation
# backgrounds.
#
# Kaleido maps 1 px -> 0.75 pt, so width=720 yields a 540 x 434.88 pt page
# = 190.5 x 153.4 mm: just over the 180 mm double-column width and well under
# the 170 mm height cap. Scaled to 180 mm every font lands in the 5-7 pt band.
from pathlib import Path

W, H = 720, 580  # must match fig.update_layout above

fig.write_image(Path('manuscript') / '2.separation.pdf',
                format='pdf', width=W, height=H)

In [34]:
# ─────────────────────────────────────────────────
# Table 1 — Peak statistics per distance metric
# ─────────────────────────────────────────────────
# Reporting convention:
#   * Only eligible layers (PERMDISP p > 0.05) are considered.
#   * PERMANOVA pseudo-F is reported at its peak eligible layer.
#   * PERMDISP F and p are reported at that same layer.

import pandas as pd

ALPHA = 0.05
P_FLOOR = 1.0 / (PERMU + 1)


def fmt_p(p):
    return f"<{P_FLOOR:.4f}" if p <= P_FLOOR + 1e-12 else f"{p:.4f}"


# Restrict to eligible layers, then find peak pseudo-F within that subset
perm_admissible = (permanova_pd
    .merge(permdisp_pd[['dist_type', 'layer', 'p-value']]
           .rename(columns={'p-value': 'permdisp_p'}),
           on=['dist_type', 'layer'])
    .query('permdisp_p > @ALPHA'))

F_peaks = (perm_admissible
    .loc[perm_admissible.groupby('dist_type')['pseudo-F'].idxmax()]
    .set_index('dist_type')[['layer', 'pseudo-F', 'p-value']])


def at_permanova_peak(df, stat_col):
    """Look up `stat_col` and p-value at each metric's eligible peak layer."""
    return (df.merge(F_peaks[['layer']].rename(columns={'layer': 'peak_layer'}),
                     left_on='dist_type', right_index=True)
    .query('layer == peak_layer')
    .set_index('dist_type')[[stat_col, 'p-value']])


permdisp_at_peak = at_permanova_peak(permdisp_pd, 'F-value')
anosim_at_peak = at_permanova_peak(anosim_pd, 'R')  # kept for supplement

# Combine stats — all evaluated at the eligible PERMANOVA peak layer
summary = pd.DataFrame({
    'L': F_peaks['layer'].astype(int),
    'PERMANOVA F': F_peaks['pseudo-F'],
    'p (PERMANOVA)': F_peaks['p-value'],
    'PERMDISP F': permdisp_at_peak['F-value'],
    'p (PERMDISP)': permdisp_at_peak['p-value'],
})

# Order rows to match Fig. 2 metric order
metric_order = ['centroid_cosine', 'centroid_cosine_unit',
                'centroid_euclidean', 'centroid_euclidean_unit',
                'earth_mover', 'earth_mover_unit',
                'energy', 'energy_unit']
summary = summary.reindex(metric_order)


def split_label(s):
    if s.endswith('_unit'):
        return (s[:-5].replace('_', ' ').title(), 'unit-normalized')
    return (s.replace('_', ' ').title(), 'raw')


summary.index = pd.MultiIndex.from_tuples(
    [split_label(m) for m in summary.index],
    names=['Metric', 'Normalization'])

display_df = summary.copy()
for col in ['p (PERMANOVA)', 'p (PERMDISP)']:
    display_df[col] = display_df[col].apply(fmt_p)

display_df.columns = pd.MultiIndex.from_tuples([
    ('', 'Layer'),
    ('PERMANOVA', 'pseudo-F'),
    ('PERMANOVA', 'p'),
    ('PERMDISP', 'F'),
    ('PERMDISP', 'p'),
])

styled = (display_df.style
.format({('PERMANOVA', 'pseudo-F'): '{:.2f}',
         ('PERMDISP', 'F'): '{:.2f}',
         ('', 'Layer'): '{:.0f}'})
.set_table_styles([
    {'selector': 'caption',
     'props': [('caption-side', 'top'),
               ('font-family', 'Helvetica, Arial, sans-serif'),
               ('font-size', '14pt'),
               ('font-weight', 'normal'),
               ('text-align', 'left'),
               ('padding', '8.4px 0 16.8px 0'),
               ('color', '#333'),
               ('line-height', '1.4')]},
    {'selector': 'table',
     'props': [('font-family', 'Helvetica, Arial, sans-serif'),
               ('font-size', '14pt'),
               ('border-collapse', 'collapse')]},
    {'selector': 'th',
     'props': [('font-family', 'Helvetica, Arial, sans-serif'),
               ('font-weight', '600'),
               ('text-align', 'center'),
               ('padding', '8.4px 16.8px'),
               ('border-bottom', '1.05px solid #555'),
               ('background-color', '#fafafa')]},
    {'selector': 'thead tr:first-child th',
     'props': [('font-weight', '700'),
               ('font-size', '14pt'),
               ('background-color', '#f3f3f3'),
               ('border-bottom', '0.7px solid #aaa')]},
    # Row dividers on data cells match the Normalization-column lines
    # (same 1.05px #555), so those horizontal rules continue across all
    # right-hand columns. The Metric column keeps its rowspan, so its only
    # divider is at each 2-row group boundary.
    {'selector': 'td',
     'props': [('text-align', 'center'),
               ('padding', '8.4px 16.8px'),
               ('border-bottom', '1.05px solid #555')]},
    {'selector': 'th.row_heading',
     'props': [('text-align', 'left'),
               ('background-color', '#fafafa'),
               ('font-weight', '500')]},
    {'selector': 'tbody tr:hover',
     'props': [('background-color', '#fffbe6')]},
]))

styled

In [35]:
# Markdown — paste directly into the manuscript draft
print(display_df.to_markdown(tablefmt='github'))

# LaTeX — for the journal submission
print(display_df.style
.format({('PERMANOVA', 'pseudo-F'): '{:.2f}',
         ('PERMDISP', 'F'): '{:.2f}',
         ('', 'Layer'): '{:.0f}'})
.to_latex(
    column_format='llrrrrr',  # 2 row-heading cols + 5 data cols
    hrules=True,
    caption=('Peak PERMANOVA and PERMDISP statistics per distance metric. '
             'PERMDISP F is evaluated at the PERMANOVA peak layer for each metric.'),
    label='tab:per-metric-peaks',
))

# CSV — for spreadsheets / collaborators
# summary.to_csv('74.table1_per_metric_peaks.csv')


|                                           |   ('', 'Layer') |   ('PERMANOVA', 'pseudo-F') | ('PERMANOVA', 'p')   |   ('PERMDISP', 'F') |   ('PERMDISP', 'p') |
|-------------------------------------------|-----------------|-----------------------------|----------------------|---------------------|---------------------|
| ('Centroid Cosine', 'raw')                |              36 |                     6.35355 | <0.0001              |            3.66419  |              0.0686 |
| ('Centroid Cosine', 'unit-normalized')    |              40 |                     6.36862 | <0.0001              |            3.76374  |              0.0832 |
| ('Centroid Euclidean', 'raw')             |              21 |                     6.9134  | 0.0002               |            1.23657  |              0.4313 |
| ('Centroid Euclidean', 'unit-normalized') |              35 |                     3.65689 | 0.0005               |            3.82858  |              0.1104 |
| ('Earth Mover', 'raw')          

## Supplement — ANOSIM

In [36]:
# Sort by peak R (descending) — may differ slightly from PERMANOVA ordering;
# uncomment the next line if you want to mirror PERMANOVA panel order instead
# sorted_dist_types_R = sorted_dist_types
sorted_dist_types_R = (anosim_pd.groupby('dist_type')['R']
                       .max()
                       .sort_values(ascending=False).index.tolist())

rows, cols = 3, 3
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[FORMAL_TITLES.get(dt, dt) for dt in sorted_dist_types_R],
    shared_xaxes=False,
    shared_yaxes=True,
    horizontal_spacing=0.04,
    vertical_spacing=0.16,
)

fig.update_layout(
    template='plotly_white',
    font=dict(family="Helvetica, Arial, sans-serif", size=10, color="black"),
    title=dict(
        text="Per-layer ANOSIM R by distance metric",
        font=dict(size=13, color="black"),
        x=0.0, xanchor='left',
        pad=dict(t=10, b=10),
    ),
    margin=dict(l=55, r=20, t=70, b=50),
    width=720, height=560,
    showlegend=False,
)

fig.update_annotations(font=dict(size=10, color="black",
                                 family="Helvetica, Arial, sans-serif"))

for i, dt in enumerate(sorted_dist_types_R):
    row = i // cols + 1
    col = i % cols + 1
    sub = anosim_pd[anosim_pd['dist_type'] == dt]

    fig.add_hline(y=0, line_dash="dash", line_color=COLOR_REF,
                  line_width=0.5, row=row, col=col)

    fig.add_trace(
        go.Scatter(
            x=sub['layer'], y=sub['R'],
            mode='lines',
            line=dict(color=COLOR_PRIMARY, width=1.25),
            customdata=sub[['p-value']],
            hovertemplate='layer=%{x}<br>R=%{y:.3f}<br>p=%{customdata[0]:.4f}<extra></extra>',
        ),
        row=row, col=col,
    )

    # Peak marker + annotation
    peak = sub.loc[sub['R'].idxmax()]
    fig.add_trace(
        go.Scatter(
            x=[peak['layer']], y=[peak['R']],
            mode='markers',
            marker=dict(color=COLOR_HIGHLIGHT, size=5),
            showlegend=False, hoverinfo='skip',
        ),
        row=row, col=col,
    )
    peak_layer = int(peak['layer'])
    peak_R = peak['R']

    # Horizontal: push annotation to the side of the panel with more empty space.
    # Peaks in left half → text on right; peaks in right half → text on left.
    ax_val = 55 if peak_layer < 31 else -55

    # Vertical: y-axis now spans [-0.05, 0.3]. Peaks near the top (R ≳ 0.19)
    # have little headroom → place text below; lower peaks → place above.
    ay_val = 16 if peak_R > 0.19 else -16

    fig.add_annotation(
        x=peak['layer'], y=peak['R'],
        text=f"R = {peak['R']:.3f}<br>L{peak_layer}",
        showarrow=True, arrowhead=2, arrowwidth=0.6, arrowsize=1,
        arrowcolor=COLOR_HIGHLIGHT,
        font=dict(size=8.5, color=COLOR_HIGHLIGHT,
                  family="Helvetica, Arial, sans-serif"),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.12)', borderwidth=0.5,
        ax=ax_val, ay=ay_val,
        row=row, col=col,
    )

for c in range(1, cols + 1):
    fig.update_xaxes(title_text="Transformer Layer",
                     title_font=dict(size=10),
                     row=rows, col=c)

fig.update_yaxes(title_text="ANOSIM R",
                 title_font=dict(size=10),
                 row=2, col=1)

fig.update_xaxes(
    tickfont=dict(size=9),
    showgrid=False,
    linecolor='black', linewidth=0.6,
    ticks='outside', ticklen=3, tickwidth=0.6,
)
fig.update_yaxes(
    range=[-0.05, 0.3],
    tickfont=dict(size=9),
    showgrid=True, gridcolor='rgba(200,200,200,0.3)', gridwidth=0.5,
    linecolor='black', linewidth=0.6,
    ticks='outside', ticklen=3, tickwidth=0.6,
)

fig.show()

In [37]:
# Supplement Table — ANOSIM R at PERMANOVA peak layer
supp_summary = pd.DataFrame({
    'L': F_peaks['layer'].astype(int),
    'ANOSIM R': anosim_at_peak['R'],
    'p (ANOSIM)': anosim_at_peak['p-value'],
}).reindex(metric_order)

supp_summary.index = pd.MultiIndex.from_tuples(
    [split_label(m) for m in supp_summary.index],
    names=['Metric', 'Normalization'])

supp_display = supp_summary.copy()
supp_display['p (ANOSIM)'] = supp_display['p (ANOSIM)'].apply(fmt_p)
supp_display.columns = pd.MultiIndex.from_tuples([
    ('', 'Layer'),
    ('ANOSIM', 'R'),
    ('ANOSIM', 'p'),
])

supp_styled = (supp_display.style
.format({('ANOSIM', 'R'): '{:.3f}', ('', 'Layer'): '{:.0f}'})
.background_gradient(subset=[('ANOSIM', 'R')], cmap='Greens', vmin=0, vmax=1)
.set_table_styles([
    {'selector': 'table',
     'props': [('font-family', 'Helvetica, Arial, sans-serif'),
               ('font-size', '10pt'), ('border-collapse', 'collapse')]},
    {'selector': 'th',
     'props': [('font-weight', '600'), ('text-align', 'center'),
               ('padding', '6px 12px'), ('border-bottom', '0.75px solid #555'),
               ('background-color', '#fafafa')]},
    {'selector': 'td',
     'props': [('text-align', 'center'), ('padding', '6px 12px'),
               ('border-bottom', '0.5px solid #eee')]},
    {'selector': 'th.row_heading',
     'props': [('text-align', 'left'), ('background-color', '#fafafa')]},
]))
supp_styled


In [38]:
# Supplement ANOSIM export
print(supp_display.to_markdown(tablefmt='github'))

print(supp_display.style
.format({('ANOSIM', 'R'): '{:.3f}', ('', 'Layer'): '{:.0f}'})
.to_latex(
    column_format='llrrr',
    hrules=True,
    caption=('Supplement: ANOSIM R statistics per distance metric, '
             'evaluated at the PERMANOVA peak layer.'),
    label='tab:supp-anosim',
))


|                                           |   ('', 'Layer') |   ('ANOSIM', 'R') | ('ANOSIM', 'p')   |
|-------------------------------------------|-----------------|-------------------|-------------------|
| ('Centroid Cosine', 'raw')                |              36 |          0.206121 | 0.0005            |
| ('Centroid Cosine', 'unit-normalized')    |              40 |          0.200345 | 0.0007            |
| ('Centroid Euclidean', 'raw')             |              21 |          0.232684 | <0.0001           |
| ('Centroid Euclidean', 'unit-normalized') |              35 |          0.180916 | 0.0014            |
| ('Earth Mover', 'raw')                    |              21 |          0.223826 | 0.0002            |
| ('Earth Mover', 'unit-normalized')        |              35 |          0.132968 | 0.0093            |
| ('Energy', 'raw')                         |              21 |          0.202063 | 0.0010            |
| ('Energy', 'unit-normalized')             |              35 | 